# 모듈 (2/5): Vector RAG — 문서 전처리부터 LangChain 조립까지
## 지식(Memory & Knowledge) 시리즈

---

[1편](M04_1_embeddings.ipynb)에서 만든 **임베딩** 위에 실제로 쓸 수 있는 **Vector RAG** 를 세웁니다.
이 노트북의 절반은 검색 이전 단계인 **문서 전처리** 에 씁니다. RAG 품질은 대부분 거기서 갈립니다.

### 학습 목표
1. **개인 문서 구조화** — 파일 → front matter + 섹션 트리로 파싱
2. **청크 분할 전략 비교** — 고정 / 재귀 / 섹션 인식 / 섹션+병합을 **검색 정확도로** 선택
3. **메타데이터 설계** — 출처 추적 · 필터링 · 권한 제어의 근거
4. **벡터 DB 3종 비교** — ChromaDB · FAISS · Qdrant 를 같은 데이터로
5. **LangChain 조립** — PromptTemplate → LCEL 체인 → RAG Agent

### 이 시리즈 구성
1. `M04_1_embeddings.ipynb` — 임베딩 · 시맨틱 유사도 · 공급자 비교
2. **`M04_2_vector_rag.ipynb`** ← (현재) 문서 전처리 · 벡터 DB 3종 · LangChain RAG
3. `M04_3_graph_rag.ipynb` — 지식 그래프 · Neo4j · LlamaIndex GraphRAG
4. `M04_4_agent_memory.ipynb` — 에이전트 메모리 **직접 구현** · Deep-Knowledge Agent
5. `M04_5_memory.ipynb` — LangGraph **메모리 인프라**(체크포인터 · Store · 영속화)

> 📦 **환경 설치·실행 명령**은 [`env_guides/M04_2_vector_rag.md`](env_guides/M04_2_vector_rag.md) 참고.
> 구현은 [`agentic_lib/doc_prep.py`](agentic_lib/doc_prep.py),
> [`agentic_lib/vector_stores.py`](agentic_lib/vector_stores.py),
> [`agentic_lib/lc_rag.py`](agentic_lib/lc_rag.py) 로 분리해 import 합니다.

### 전체 파이프라인

```
 [개인 문서 .md]                              ← 1장: 전처리
       │  front matter + 섹션 파싱
       ▼
 [PersonalDoc]  title/category/tags/sections
       │  청크 분할 (4가지 전략 비교 → 검색 정확도로 선택)
       ▼
 [Chunk + 메타데이터]  doc_id·section·category·created …
       │  임베딩 (M04_1 의 get_embedder)
       ▼
 ┌──────────┬──────────┬──────────┐              ← 2장: 벡터 DB
 │ ChromaDB │  FAISS   │  Qdrant  │  같은 벡터, 같은 질의
 └────┬─────┴────┬─────┴────┬─────┘
      └──────────┼──────────┘
                 ▼
        [검색 결과 = 근거 청크]
                 │
      ┌──────────┴───────────┐                   ← 3~4장: LangChain
      ▼                      ▼
 [LCEL 체인]             [RAG Agent]
 검색 1회 → 답변          검색 횟수를 LLM 이 결정
```

---
## 0. 준비

### 필요 패키지 (CMD)
```bat
REM 문서 전처리 + 임베딩
uv pip install sentence-transformers numpy langchain langchain-text-splitters

REM 벡터 DB 3종
uv pip install chromadb faiss-cpu qdrant-client
```

> LLM 은 `.env` 의 `LLM_PROVIDER` 를 그대로 씁니다(`utils.get_llm()`).
> **4장 RAG Agent 는 도구 호출을 지원하는 모델** 이 필요합니다
> (ollama `qwen3:8b`, google, nvidia, openrouter 모두 가능).

In [1]:
# 자기완결 setup: 이 노트북만 단독 실행해도 되도록 공통 셋업을 맨 앞에 둔다
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()  # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import uv_install, get_llm, test_llm_connection, LLM_PROVIDER
# 공통 라이브러리
from agentic_lib import bootstrap, embeddings as emb   # 임베딩 공급자(M04_1)
from agentic_lib import doc_prep                       # 문서 구조화·청킹·메타데이터
from agentic_lib import vector_stores as vs            # Chroma/FAISS/Qdrant 어댑터
from agentic_lib import lc_rag                         # PromptTemplate·LCEL·Agent
from agentic_lib.bootstrap import to_text

uv_install(['sentence-transformers', 'numpy', 'langchain',
            'chromadb', 'faiss-cpu', 'qdrant-client'])

llm = test_llm_connection()  # 공급자 무관

LLM 공급자: openrouter
  OpenRouter Key: 설정됨  /  Model: nvidia/nemotron-3-super-120b-a12b:free


[uv] 설치 완료: ['sentence-transformers', 'numpy', 'langchain', 'chromadb', 'faiss-cpu', 'qdrant-client']


LLM 연결 성공 [openrouter]: 1 + 1 = 2.


---
## 1. 개인 문서의 구조화

RAG 실습은 보통 "문자열 리스트"에서 시작하지만, 현실의 문서는 **디스크의 파일** 입니다.
여기서는 개인 지식 관리(PKM)에서 흔한 형태 — 회의록·업무보고·학습노트·규정·개인메모를
마크다운 파일로 만들어 놓고 시작합니다.

각 파일 맨 위에는 **front matter** 로 메타데이터를 적어 둡니다.

```markdown
---
title: 사내 문서 검색 시스템 킥오프 회의록
category: 회의록
tags: [프로젝트, 킥오프, RAG, 일정]
created: 2026-03-14
author: 신성태
---

# 사내 문서 검색 시스템 킥오프 회의록

## 참석자
...
## 결정 사항
...
```

이 구조를 살려서 읽으면 **제목·분류·작성일·섹션** 을 청크 메타데이터로 물려줄 수 있습니다.
통짜 텍스트로 읽으면 이 정보가 전부 사라집니다.

In [2]:
# 실습용 개인 문서를 실제 파일로 생성한다 (notebooks/workspace/personal_docs/)
DOC_DIR = os.path.join("workspace", "personal_docs")
paths = doc_prep.write_sample_docs(DOC_DIR)

print(f"문서 {len(paths)}개 생성 → {DOC_DIR}\n")
for path in paths:
    print(f"  {os.path.getsize(path):>6,} bytes  {os.path.basename(path)}")

# 원본 파일이 실제로 어떻게 생겼는지 첫 파일의 앞부분을 그대로 본다
print("\n" + "=" * 78)
print(f"[원본 미리보기] {os.path.basename(paths[0])}")
print("=" * 78)
with open(paths[0], encoding="utf-8") as f:
    print("".join(f.readlines()[:14]))

문서 6개 생성 → workspace\personal_docs

   1,191 bytes  2026-03-14_회의록_프로젝트킥오프.md
   1,163 bytes  학습노트_langchain_lcel.md
     903 bytes  업무_주간보고_2026-W12.md
     740 bytes  여행_오사카_준비메모.md
     648 bytes  건강_러닝기록_3월.md
     749 bytes  규정_문서보안_지침.md

[원본 미리보기] 2026-03-14_회의록_프로젝트킥오프.md
---
title: 사내 문서 검색 시스템 킥오프 회의록
category: 회의록
tags: [프로젝트, 킥오프, RAG, 보안]
created: 2026-03-14
author: 신성태
---

# 사내 문서 검색 시스템 킥오프 회의록

## 참석자
개발팀 4명, 기획팀 2명이 참석했다. 외부 자문으로 데이터 플랫폼팀 1명이 배석했다.

## 배경



In [3]:
# 파일 → 구조화된 PersonalDoc (front matter 파싱 + 섹션 분할)
docs = doc_prep.load_markdown_docs(DOC_DIR)
doc_prep.print_doc_structure(docs)

# 섹션 트리 확인 — 이 구조가 나중에 청크 메타데이터의 'section' 이 된다
print("\n=== 섹션 트리 ===")
for doc in docs[:3]:
    print(f"\n{doc.doc_id}")
    for section in doc.sections:
        preview = section["text"].replace("\n", " ")[:46]
        print(f"  ├ {section['heading']:<16} ({len(section['text']):>3}자)  {preview}…")

문서 ID                              분류       작성일            글자수   섹션  태그
------------------------------------------------------------------------------------------------------------
2026-03-14_회의록_프로젝트킥오프             회의록      2026-03-14     439    4  프로젝트, 킥오프, RAG, 보안
건강_러닝기록_3월                         개인메모     2026-03-31     233    3  운동, 러닝, 건강
규정_문서보안_지침                         사내규정     2025-11-05     259    3  보안, 규정, 권한
업무_주간보고_2026-W12                   업무보고     2026-03-20     328    3  프로젝트, 주간보고, RAG
여행_오사카_준비메모                        개인메모     2026-01-11     259    3  여행, 오사카, 체크리스트
학습노트_langchain_lcel                학습노트     2026-02-28     516    4  LangChain, LCEL, RAG
------------------------------------------------------------------------------------------------------------
총 6개 문서 / 20개 섹션 / 2,034자

=== 섹션 트리 ===

2026-03-14_회의록_프로젝트킥오프
  ├ 참석자              ( 48자)  개발팀 4명, 기획팀 2명이 참석했다. 외부 자문으로 데이터 플랫폼팀 1명이 배석했…
  ├ 배경               (105자)  사내 위키와 공유 드라이브에 문서가 흩어져 있어 필요한 정

### 1-2. 청크 분할 — 네 가지 전략

청크는 **검색의 최소 단위** 입니다. 크기 선택에는 상충이 있습니다.

| 청크가 너무 크면 | 청크가 너무 작으면 |
|---|---|
| 관련 없는 내용까지 딸려 와 LLM 이 헷갈린다 | 맥락이 끊겨 답을 만들 수 없다 |
| 컨텍스트 토큰을 낭비한다 | 파편이 상위 k 자리를 차지해 정작 필요한 청크를 밀어낸다 |

네 가지 전략을 같은 문서에 적용해 비교합니다.

| 전략 | 방식 | 섹션 메타데이터 |
|---|---|---|
| `fixed` | 글자 수로 기계적으로 자름 | ❌ 잃음 |
| `recursive` | 문단 → 문장 → 단어 순으로 자연 경계를 찾음 | ❌ 잃음 |
| `header` | 섹션 경계를 먼저 존중, 긴 섹션만 재귀 분할 | ✅ 보존 |
| `header_merged` | header + **짧은 섹션을 앞과 병합** | ✅ 보존 |

In [4]:
# 네 전략을 같은 문서에 적용하고 청크 통계를 비교한다
chunk_sets = doc_prep.compare_chunking(docs, chunk_size=400, overlap=80)

전략                청크 수     평균길이     최소     최대       문장중간절단  설명
--------------------------------------------------------------------------------------------------------------
고정 크기                9      245      8    400         22%  글자 수로만 자름 — 구현은 쉽지만 단어·문장이 끊긴다
재귀 분할                8      253     70    366          0%  문단→문장→단어 순으로 경계를 찾음 — 범용 기본값
섹션 인식               20       85     33    163          0%  섹션 경계 존중 — 맥락은 좋지만 짧은 섹션이 파편으로 남는다
섹션+병합 ⭐             13      132     33    230          0%  짧은 섹션을 앞과 병합 — 파편 제거, 실무 권장값
--------------------------------------------------------------------------------------------------------------
※ '문장중간절단' = 청크가 문장부호로 끝나지 않는 비율(낮을수록 자연스러운 분할)
※ 평균길이가 너무 짧으면(<150자) 검색은 되어도 맥락이 없어 답변 품질이 떨어진다


**통계만으로는 고를 수 없습니다.** 청크 수가 적다고 좋은 것도, 많다고 좋은 것도 아닙니다.
결국 판단 기준은 하나 — **질문을 던졌을 때 정답 대목이 상위에 오는가.**

그래서 골드셋(질문 10개 + 정답 문서·섹션)으로 **검색 정확도를 직접 측정** 합니다.

In [5]:
# 골드셋: (질문, 정답 문서, 정답 섹션) — 질문과 문서가 단어를 공유하지 않게 만들었다
print(f"골드셋 질문 {len(doc_prep.GOLD_QUESTIONS)}개 (일부)\n")
for question, gold_doc, gold_section in doc_prep.GOLD_QUESTIONS[:4]:
    print(f"  Q. {question}")
    print(f"     → 정답: {gold_doc} › {gold_section}\n")

# 전략별로 실제 검색 정확도를 잰다 (같은 임베더 · 같은 벡터 DB)
embedder = emb.get_embedder("local")
print(f"임베더: {embedder.name} ({embedder.dim}차원)\n")

report = vs.evaluate_chunking(chunk_sets, embedder, doc_prep.GOLD_QUESTIONS, k=3)

골드셋 질문 10개 (일부)

  Q. 킥오프 회의에서 결정된 것 중 권한 관련 내용은?
     → 정답: 2026-03-14_회의록_프로젝트킥오프 › 결정 사항

  Q. 프로토타입 데모까지 기간이 얼마나 되지?
     → 정답: 2026-03-14_회의록_프로젝트킥오프 › 결정 사항

  Q. LCEL 에서 자주 하는 실수는?
     → 정답: 학습노트_langchain_lcel › 실수했던 것

  Q. RunnableLambda 는 언제 쓰나?
     → 정답: 학습노트_langchain_lcel › 자주 쓰는 조각



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

임베더: local:paraphrase-multilingual-MiniLM-L12-v2 (384차원)

전략                          청크     평균길이   문서 Top-1   섹션 Top-1   섹션 Top-3
----------------------------------------------------------------------------


fixed                        9      245      100%     구조없음     구조없음


recursive                    8      253      100%     구조없음     구조없음


header                      20       85      100%       70%       80%


header_merged               13      132      100%       80%       90%
----------------------------------------------------------------------------
※ '구조없음' = 섹션 메타데이터를 만들지 못하는 전략(고정·재귀 분할).
   문서는 맞게 찾아도 **어느 대목인지 짚어 줄 수 없어** 출처 표시·재확인이 어렵다.


**측정이 알려 준 것**

- 어느 전략이든 **문서** 는 잘 찾습니다(Top-1 100%). 문서가 6개뿐이라 쉬운 문제입니다.
- 갈리는 것은 **문서 안에서 어느 대목인지** 입니다.
  `fixed`/`recursive` 는 섹션 정보를 아예 만들지 못해 "이 문서 어딘가"까지만 답할 수 있습니다.
- `header` 는 섹션을 보존하지만 짧은 섹션이 파편으로 남아 상위 k 를 잡아먹습니다.
- `header_merged` 가 파편을 없애 **섹션 Top-1 을 끌어올립니다** → 이 전략을 채택합니다.

> 실무 감각: 청크 평균 길이가 **150자 미만** 이면 대개 너무 잘게 쪼갠 것입니다.
> 검색은 되는데 답변이 부실하다면 청크 크기부터 의심하세요.

### 1-3. 메타데이터 설계

메타데이터는 "있으면 좋은 것"이 아니라 **RAG 를 실제로 쓸 수 있게 만드는 장치** 입니다.

| 목적 | 필요한 필드 | 없으면 생기는 일 |
|---|---|---|
| **출처 표시** | `doc_id`, `title`, `section`, `source` | 답변을 검증할 수 없다(환각과 구별 불가) |
| **필터링** | `category`, `tags`, `created` | 부서·기간·종류로 좁힐 수 없다 |
| **권한 제어** | `category`(또는 등급) | 볼 수 없는 문서가 검색 결과에 뜬다 |
| **원문 되짚기** | `chunk_index`, `chunk_total` | 앞뒤 맥락을 다시 읽을 수 없다 |

**설계 규칙 하나**: 값은 반드시 **스칼라**(문자열·숫자·불리언)여야 합니다.
대부분의 벡터 DB 는 리스트를 메타데이터로 받지 못합니다 →
`tags: [a, b]` 는 `tags: "a,b"` 로 평탄화합니다.

In [6]:
# 채택한 전략으로 최종 청킹 + 메타데이터 확인
chunks = chunk_sets["header_merged"]
print(f"최종 청크 {len(chunks)}개\n")

# 청크 하나의 메타데이터 스키마를 그대로 펼쳐 본다
sample = chunks[0]
print("=== 청크 메타데이터 스키마 ===")
for key, value in sample.metadata.items():
    print(f"  {key:<13} = {value!r:<44} ({type(value).__name__})")
print(f"\n  chunk_id      = {sample.chunk_id!r}   ← 벡터 DB 의 기본키")

print("\n" + "=" * 78)
doc_prep.print_chunk_samples(chunks, n=3)

최종 청크 13개

=== 청크 메타데이터 스키마 ===
  doc_id        = '2026-03-14_회의록_프로젝트킥오프'                     (str)
  title         = '사내 문서 검색 시스템 킥오프 회의록'                       (str)
  category      = '회의록'                                        (str)
  author        = '신성태'                                        (str)
  created       = '2026-03-14'                                 (str)
  tags          = '프로젝트,킥오프,RAG,보안'                            (str)
  section       = '참석자 / 배경'                                   (str)
  chunk_index   = 0                                            (int)
  chunk_total   = 3                                            (int)
  source        = '2026-03-14_회의록_프로젝트킥오프.md'                  (str)

  chunk_id      = '2026-03-14_회의록_프로젝트킥오프#0'   ← 벡터 DB 의 기본키

[2026-03-14_회의록_프로젝트킥오프#0] 사내 문서 검색 시스템 킥오프 회의록 › 참석자 / 배경
  분류=회의록 | 태그=프로젝트,킥오프,RAG,보안 | 작성일=2026-03-14
  개발팀 4명, 기획팀 2명이 참석했다. 외부 자문으로 데이터 플랫폼팀 1명이 배석했다.  사내 위키와 공유 드라이브에 문서가 흩어져 있어 필요한 정보를 찾는 데 …

[2026-03-14_회의

---
## 2. 벡터 DB 3종 — ChromaDB · FAISS · Qdrant

같은 청크·같은 임베딩을 세 가지 백엔드에 넣고 똑같이 검색합니다.
임베딩을 **바깥에서 주입** 하므로 세 DB 는 완전히 동일한 벡터를 다룹니다 —
그래야 비교가 공정하고, 차이가 나면 그건 DB 자체의 특성입니다.

| | ChromaDB | FAISS | Qdrant |
|---|---|---|---|
| 정체 | 벡터 **DB** | 벡터 **인덱스 라이브러리** | 벡터 **DB** (Rust) |
| 만든 곳 | Chroma | Meta | Qdrant |
| 메타데이터 | 내장 | ❌ 직접 관리 | 내장 |
| 필터링 | 인덱스 통합 | **후처리(post-filter)** | 인덱스 통합 |
| 영속화 | `PersistentClient` | 파일 저장 직접 구현 | 서버 / 로컬 파일 |
| 서버 필요 | ❌ | ❌ | ❌ (`:memory:` 모드) |
| 강점 | 가장 간단, 프로토타입 | **최고 속도**, 대규모 | 필터 성능, 운영 기능 |

---
- **FAISS (Facebook AI Similarity Search)**
: Meta(Facebook)에서 개발한 순수 벡터 검색 라이브러리. 엄밀히 말해 데이터베이스가 아님.
: 메모리(RAM) 기반으로 작동하며 GPU 가속을 지원하여 수억에서 수십억 개의 벡터를 검색할 때 가장 빠른 속도를 자랑함.
- **Chroma (ChromaDB)**
: AI 개발자들이 LLM 애플리케이션을 빠르게 만들 수 있도록 돕는 데 초점을 맞춘 오픈소스 벡터 데이터베이스. 인메모리나 로컬 SQLite 기반으로 즉시 실행 가능. LangChain이나 LlamaIndex 같은 프레임워크와 매우 매끄럽게 연동됨.
- **Qdrant**
: Rust 언어로 개발된 상용 레벨의 고성능 벡터 데이터베이스. 강력한 필터링과 유연성

💡 **요약**:오늘 당장 RAG 튜토리얼을 따라 해보고 싶다면 Chroma를, 수억 개의 데이터를 0.01초라도 빨리 검색해야 한다면 FAISS를, 안정적으로 실제 서비스를 운영해야 한다면 Qdrant를 선택 추천.

In [7]:
# 설치된 백엔드 점검 후 세 곳에 같은 청크를 적재한다
available = vs.available_stores()

stores = []
print()
for kind in ["chroma", "faiss", "qdrant"]:
    if not available.get(kind):
        print(f"[{kind}] 미설치 — 건너뜁니다")
        continue
    store = vs.get_vector_store(kind, embedder, collection="personal_docs")
    elapsed = store.add(chunks)
    stores.append(store)
    print(f"[{kind:<7}] 적재 {store.count():>3}건  {elapsed:.2f}s")

print(f"\n비교 대상 {len(stores)}종 (임베더는 {embedder.name} 하나로 공유)")

백엔드        사용가능     설치 명령                              특징
--------------------------------------------------------------------------------------------------------
chroma     ✅        uv pip install chromadb            임베딩 함수 내장 · 파일 영속화 쉬움 · 프로토타입 기본값
faiss      ✅        uv pip install faiss-cpu           DB 가 아닌 인덱스 · 가장 빠름 · 메타데이터는 직접 관리
qdrant     ✅        uv pip install qdrant-client       메타데이터 필터가 인덱스에 통합 · :memory: 모드 지원



[chroma ] 적재  13건  0.17s
[faiss  ] 적재  13건  0.10s


[qdrant ] 적재  13건  0.11s

비교 대상 3종 (임베더는 local:paraphrase-multilingual-MiniLM-L12-v2 하나로 공유)


In [8]:
# 같은 질의를 세 백엔드에 던져 결과·속도를 나란히 본다
vs.compare_search(stores, "청크를 잘게 쪼갰을 때 생긴 문제는?", k=2)

질의: 청크를 잘게 쪼갰을 때 생긴 문제는?
[chroma]  17.8 ms  (2건)
   1. [0.346] 2026년 12주차 주간 업무 보고 › 이번 주 한 일 / 문제와 해결
      문서 수집 파이프라인의 첫 버전을 만들었다. 마크다운과 텍스트 파일을 읽어 front matter 를 분리하고 섹션 단위로 쪼개는 데까지…
   2. [0.262] LangChain LCEL 학습 노트 › 자주 쓰는 조각 / 실수했던 것
      RunnablePassthrough 는 입력을 그대로 흘려보내면서 곁가지로 다른 값을 덧붙일 때 쓴다. RunnableLambda 는 평…

[faiss]  15.5 ms  (2건)
   1. [0.346] 2026년 12주차 주간 업무 보고 › 이번 주 한 일 / 문제와 해결
      문서 수집 파이프라인의 첫 버전을 만들었다. 마크다운과 텍스트 파일을 읽어 front matter 를 분리하고 섹션 단위로 쪼개는 데까지…
   2. [0.262] LangChain LCEL 학습 노트 › 자주 쓰는 조각 / 실수했던 것
      RunnablePassthrough 는 입력을 그대로 흘려보내면서 곁가지로 다른 값을 덧붙일 때 쓴다. RunnableLambda 는 평…

[qdrant]  17.3 ms  (2건)
   1. [0.346] 2026년 12주차 주간 업무 보고 › 이번 주 한 일 / 문제와 해결
      문서 수집 파이프라인의 첫 버전을 만들었다. 마크다운과 텍스트 파일을 읽어 front matter 를 분리하고 섹션 단위로 쪼개는 데까지…
   2. [0.262] LangChain LCEL 학습 노트 › 자주 쓰는 조각 / 실수했던 것
      RunnablePassthrough 는 입력을 그대로 흘려보내면서 곁가지로 다른 값을 덧붙일 때 쓴다. RunnableLambda 는 평…



In [9]:
# 세 백엔드가 정말 같은 결과를 내는지 여러 질의로 교차 확인
vs.rank_agreement(stores, [q for q, _, _ in doc_prep.GOLD_QUESTIONS[:6]], k=3)

질의                                 Top-1 청크 id                                  일치
----------------------------------------------------------------------------------------
킥오프 회의에서 결정된 것 중 권한 관련 내용은?        2026-03-14_회의록_프로젝트킥오프#2                     ✅
프로토타입 데모까지 기간이 얼마나 되지?             2026-03-14_회의록_프로젝트킥오프#1                     ✅


LCEL 에서 자주 하는 실수는?                 학습노트_langchain_lcel#0                        ✅
RunnableLambda 는 언제 쓰나?            학습노트_langchain_lcel#1                        ✅


청크 크기 문제를 어떻게 해결했나?                업무_주간보고_2026-W12#0                           ✅
3월에 총 몇 킬로미터 달렸나?                  건강_러닝기록_3월#0                                 ✅

※ 세 백엔드가 같은 벡터·같은 코사인 척도를 쓰므로 Top-1 은 일치해야 정상이다.
※ 어긋난다면 임베딩이 다르거나 거리 척도(cosine/L2) 설정이 다르다는 신호다.


### 2-2. 메타데이터 필터링 — 여기서 차이가 난다

검색 결과가 같다면 무엇으로 고를까요. **필터링 방식** 이 첫 번째 갈림길입니다.

- **Chroma / Qdrant**: 필터가 인덱스 단계에 통합 → 조건에 맞는 것 중에서 상위 k 개를 채워 준다
- **FAISS**: 벡터 인덱스일 뿐이라 필터 개념이 없음 → 넉넉히 뽑아 놓고 **버리는 방식**

아래에서 `category="사내규정"` 으로 좁혀 보면 셋 다 멀쩡히 2건을 채웁니다.
하지만 FAISS 가 2건을 채운 것은 **뒤에서 몰래 k 의 10배를 뽑고 있기 때문** 입니다.
이어지는 2-2-1 에서 그 배수를 직접 낮춰 무슨 일이 벌어지는지 확인합니다.

In [10]:
# 분류 필터를 걸고 세 백엔드를 비교
vs.compare_search(stores, "문서를 외부로 보내면 어떻게 되나?", k=2,
                  where={"category": "사내규정"})

# chroma 또는 qdrant 를 이용해 '필터를 권한으로 쓰는 방법' 설명
perm_store = next((s for s in stores if s.name in ("chroma", "qdrant")), stores[0])

print("=" * 100)
print(f"[권한 시나리오] '개인메모'를 제외하고 업무 문서만 검색  (백엔드: {perm_store.name})\n")
for category in ["회의록", "업무보고", "사내규정"]:
    hits = perm_store.search("문서 검색 시스템 관련 내용", k=1,
                             where={"category": category})
    for hit in hits:
        print(f"  {category:<8} → {hit['metadata']['title']} › {hit['metadata']['section']}")

질의: 문서를 외부로 보내면 어떻게 되나?   (필터: {'category': '사내규정'})
[chroma]  16.0 ms  (2건)
   1. [0.560] 사내 문서 보안 지침 › 위반 시
      대외비 문서를 외부로 전송하면 즉시 보고 대상이며 감사 기록이 남는다.
   2. [0.328] 사내 문서 보안 지침 › 등급 구분 / 검색 시스템 적용 원칙
      문서는 공개, 사내한정, 대외비 세 등급으로 나눈다. 등급은 문서를 만든 사람이 지정하며 지정하지 않으면 사내한정으로 간주한다.  검색 …

[faiss]  12.9 ms  (2건)
   1. [0.560] 사내 문서 보안 지침 › 위반 시
      대외비 문서를 외부로 전송하면 즉시 보고 대상이며 감사 기록이 남는다.
   2. [0.328] 사내 문서 보안 지침 › 등급 구분 / 검색 시스템 적용 원칙
      문서는 공개, 사내한정, 대외비 세 등급으로 나눈다. 등급은 문서를 만든 사람이 지정하며 지정하지 않으면 사내한정으로 간주한다.  검색 …

[qdrant]  14.2 ms  (2건)
   1. [0.560] 사내 문서 보안 지침 › 위반 시
      대외비 문서를 외부로 전송하면 즉시 보고 대상이며 감사 기록이 남는다.
   2. [0.328] 사내 문서 보안 지침 › 등급 구분 / 검색 시스템 적용 원칙
      문서는 공개, 사내한정, 대외비 세 등급으로 나눈다. 등급은 문서를 만든 사람이 지정하며 지정하지 않으면 사내한정으로 간주한다.  검색 …

[권한 시나리오] '개인메모'를 제외하고 업무 문서만 검색  (백엔드: chroma)

  회의록      → 사내 문서 검색 시스템 킥오프 회의록 › 참석자 / 배경


  업무보고     → 2026년 12주차 주간 업무 보고 › 이번 주 한 일 / 문제와 해결
  사내규정     → 사내 문서 보안 지침 › 등급 구분 / 검색 시스템 적용 원칙


위 결과만 보면 **세 백엔드가 똑같아 보입니다.** 필터를 걸어도 셋 다 2건을 잘 채웠으니까요.
하지만 FAISS 가 그 2건을 채운 방식은 나머지 둘과 완전히 다릅니다 — 그리고 그 차이는
데이터가 커지는 순간 드러납니다. 직접 열어서 확인해 봅시다.

### 2-2-1. FAISS 는 왜 k 의 10배를 뽑아야 하나

[`FaissStore.search()`](agentic_lib/vector_stores.py) 의 핵심은 이 한 줄입니다.

```python
fetch = min(len(self._payloads), k * overfetch if where else k)   # overfetch 기본값 10
```

FAISS 는 **필터가 무엇인지 모른 채** 상위 `fetch` 개를 뽑고, 그 다음에 조건에 안 맞는 것을 버립니다.
따라서 결과가 채워지느냐는 오직 하나에 달려 있습니다 —
**조건을 통과하는 청크가 전체 순위에서 몇 등에 있는가.**

- 정답이 4위에 있는데 2개만 뽑으면 → 못 건진다
- 정답이 12위에 있는데 10개만 뽑으면 → 여전히 못 건진다

`overfetch` 를 1로 낮추면 과다인출 없이 딱 `k` 개만 뽑습니다.
배수를 `1 → 2 → 5 → 10` 으로 올려 가며 **결과가 몇 건이나 돌아오는지** 세어 봅니다.

In [11]:
# FAISS 스토어와 비교용 스토어(필터가 인덱스에 통합된 쪽)를 이름으로 집는다
faiss_store = next(s for s in stores if s.name == "faiss")
indexed_store = next((s for s in stores if s.name != "faiss"), None)

# [약한 사례] 정답이 1위·4위 — 배수를 조금만 낮춰도 하나가 빠진다
vs.faiss_overfetch_demo(faiss_store, "문서를 외부로 보내면 어떻게 되나?",
                        where={"category": "사내규정"}, k=2,
                        compare_store=indexed_store)

# [심한 사례] 정답이 4위·12위 — 같은 필터, 질의만 바꿨는데 ×1 에서 결과가 아예 0건이 된다
print("\n" + "#" * 100 + "\n")
vs.faiss_overfetch_demo(faiss_store, "프로젝트 일정은?",
                        where={"category": "사내규정"}, k=2,
                        compare_store=indexed_store)

질의: 문서를 외부로 보내면 어떻게 되나?
필터: {'category': '사내규정'}   요청 k=2   전체 청크 13개
[1단계] 필터를 빼고 전체 순위를 본다 — 조건에 맞는 청크가 몇 등인가
    순위      점수  분류       통과    제목 › 섹션
  ------------------------------------------------------------------------------------------------
     1   0.560  사내규정     ✅     사내 문서 보안 지침 › 위반 시
     2   0.408  업무보고     ❌     2026년 12주차 주간 업무 보고 › 이번 주 한 일 / 문제와 해결
     3   0.385  회의록      ❌     사내 문서 검색 시스템 킥오프 회의록 › 다음 액션
     4   0.328  사내규정     ✅     사내 문서 보안 지침 › 등급 구분 / 검색 시스템 적용 원칙

  → 조건을 통과하는 청크는 2개, 전체 순위로 [1, 4] 위.
     k=2 를 채우려면 최소 **4개** 를 뽑아야 한다(= 4위까지 훑어야 한다).

[2단계] overfetch 배수를 바꿔 가며 같은 검색을 반복한다
     배수     뽑는 개수        결과  상태
  ------------------------------------------------------------------------------------------------
     ×1        2개     1건/2건  ⚠️ 2건 중 1건만 — 조용히 누락된다(오류도 안 난다)


     ×2        4개     2건/2건  ✅ 요청한 만큼 채움
     ×5       10개     2건/2건  ✅ 요청한 만큼 채움
    ×10       13개     2건/2건  ✅ 요청한 만큼 채움 (전체를 다 훑음 — 인덱스의 의미가 사라진 상태)

[3단계] 같은 조건을 chroma 에 던지면
  2건/2건 — 배수라는 개념 자체가 없다. 조건에 맞는 것 중에서 상위 k개를 인덱스가 직접 채워 준다.

####################################################################################################

질의: 프로젝트 일정은?
필터: {'category': '사내규정'}   요청 k=2   전체 청크 13개
[1단계] 필터를 빼고 전체 순위를 본다 — 조건에 맞는 청크가 몇 등인가
    순위      점수  분류       통과    제목 › 섹션
  ------------------------------------------------------------------------------------------------
     1   0.578  회의록      ❌     사내 문서 검색 시스템 킥오프 회의록 › 다음 액션
     2   0.551  회의록      ❌     사내 문서 검색 시스템 킥오프 회의록 › 결정 사항
     3   0.225  개인메모     ❌     오사카 여행 준비 메모 › 일정 / 예약할 것
     4   0.172  사내규정     ✅     사내 문서 보안 지침 › 위반 시
     …
    12   0.011  사내규정     ✅     사내 문서 보안 지침 › 등급 구분 / 검색 시스템 적용 원칙

  → 조건을 통과하는 청크는 2개, 전체 순위로 [4, 12] 위.
     k=2 를 채우려면 최소 **12개** 를 뽑아야 한다(= 12위까지 훑어야 한다).

[2단계] overfetch 배수를 바꿔 

     ×2        4개     1건/2건  ⚠️ 2건 중 1건만 — 조용히 누락된다(오류도 안 난다)
     ×5       10개     1건/2건  ⚠️ 2건 중 1건만 — 조용히 누락된다(오류도 안 난다)
    ×10       13개     2건/2건  ✅ 요청한 만큼 채움 (전체를 다 훑음 — 인덱스의 의미가 사라진 상태)

[3단계] 같은 조건을 chroma 에 던지면
  2건/2건 — 배수라는 개념 자체가 없다. 조건에 맞는 것 중에서 상위 k개를 인덱스가 직접 채워 준다.


**실험이 알려 준 것**

| 배수 | 뽑는 개수 | "문서를 외부로…" (정답 1·4위) | "프로젝트 일정은?" (정답 4·12위) |
|---|---|---|---|
| ×1 | 2개 | ⚠️ 1건만 | ❌ **0건** |
| ×2 | 4개 | ✅ 2건 | ⚠️ 1건만 |
| ×5 | 10개 | ✅ 2건 | ⚠️ 1건만 |
| ×10 | 13개(전체) | ✅ 2건 | ✅ 2건 |

세 가지를 짚고 넘어가야 합니다.

1. **10배는 근거 있는 수가 아니라 경험칙입니다.** 같은 필터·같은 k 인데 질의만 바꿨더니
   필요한 배수가 2배에서 10배로 뛰었습니다. 데이터가 바뀌면 또 달라집니다.
   즉 **안전한 배수를 미리 정할 방법이 없습니다.**
2. **모자라도 오류가 나지 않습니다.** `×1` 에서 0건이 돌아온 것은 "사내규정 문서에 그런 내용이 없다"가
   아니라 "덜 뽑아서 못 봤다"입니다. 둘은 호출한 쪽에서 **구분할 수 없습니다** — RAG 에서는 곧바로
   "제공된 문서에서 찾을 수 없습니다"라는 잘못된 답으로 이어집니다.
3. **배수를 키우면 인덱스를 쓰는 의미가 사라집니다.** 이 실습은 청크가 13개뿐이라 `×10` 이 곧 전수 검색입니다.
   1,000만 건에서 `k=10` 에 `×10` 이면 100건 스캔으로 끝나지만, 필터가 1%만 통과시키는 조건이라면
   1,000건을 뽑아야 겨우 10건을 채웁니다. **필터가 강할수록 FAISS 의 속도 이점이 상쇄됩니다.**

> 정리하면 — FAISS 의 "최고 속도"는 **필터 없는 검색** 에서의 이야기입니다.
> 메타데이터 필터가 핵심인 서비스라면 Qdrant 처럼 필터가 인덱스에 통합된 백엔드를 쓰거나,
> FAISS 를 쓰되 **필터 조건별로 인덱스를 분리**(예: 분류마다 별도 인덱스)하는 설계가 필요합니다.

### 2-3. 무엇을 고를 것인가

| 상황 | 추천 | 이유 |
|---|---|---|
| 강의·프로토타입·수천 건 | **ChromaDB** | 설치 한 줄, 메타데이터 내장, 영속화 쉬움 |
| 수백만 건 · 최고 속도 | **FAISS** | C++ 구현, GPU 지원. 메타데이터는 별도 DB 로 관리 |
| 필터가 복잡한 운영 서비스 | **Qdrant** | 필터가 인덱스에 통합, 스냅샷·복제 등 운영 기능 |

**공통 주의사항**

1. **인덱스와 질의의 임베딩 모델은 반드시 동일** — 바꾸면 전체 재색인입니다.
2. **거리 척도를 확인** — 이 노트북은 셋 다 코사인으로 맞췄습니다.
   기본값이 L2 인 백엔드에서 정규화하지 않은 벡터를 쓰면 결과가 달라집니다.
3. **인메모리는 실습용** — 커널을 재시작하면 사라집니다.
   운영에서는 Chroma `PersistentClient`, FAISS `write_index`, Qdrant 서버를 씁니다.

---
## 3. LangChain 으로 조립하기 — PromptTemplate → LCEL

검색기가 준비됐으니 이제 "검색 결과 + 질문 → 답변" 을 만듭니다.

### 3-1. PromptTemplate — 프롬프트를 '틀'로 다루기

f-string 으로 프롬프트를 만들면 당장은 편하지만, 변수가 무엇인지 코드 전체를 읽어야 알 수 있고
재사용·교체가 어렵습니다. `ChatPromptTemplate` 은 프롬프트를 **입력 변수가 선언된 객체** 로 만듭니다.

In [12]:
# RAG 용 프롬프트 템플릿 (system + human 2단 구성)
prompt = lc_rag.build_rag_prompt()

print("=== 템플릿 입력 변수 ===")
print(f"  {prompt.input_variables}\n")

print("=== 시스템 프롬프트 ===")
print(lc_rag.RAG_SYSTEM_PROMPT)

print("\n=== 변수를 채우면 이런 메시지가 만들어진다 ===")
filled = prompt.invoke({"context": "[1] (3월 러닝 기록 › 총량) 누적 거리는 96킬로미터였다.",
                        "question": "3월에 얼마나 달렸어?"})
for message in filled.messages:
    print(f"\n--- {type(message).__name__} ---")
    print(message.content[:300])

=== 템플릿 입력 변수 ===
  ['context', 'question']

=== 시스템 프롬프트 ===
당신은 사용자의 개인 문서를 검색해 답하는 비서입니다.

규칙:
1. 반드시 아래 '문서 발췌'에 있는 내용만 근거로 답하세요.
2. 발췌에 답이 없으면 "제공된 문서에서 찾을 수 없습니다"라고 솔직히 말하세요.
3. 답변 끝에 근거로 삼은 문서를 [출처: 문서제목 › 섹션] 형식으로 표시하세요.
4. 한국어로 간결하게 답하세요.

=== 변수를 채우면 이런 메시지가 만들어진다 ===

--- SystemMessage ---
당신은 사용자의 개인 문서를 검색해 답하는 비서입니다.

규칙:
1. 반드시 아래 '문서 발췌'에 있는 내용만 근거로 답하세요.
2. 발췌에 답이 없으면 "제공된 문서에서 찾을 수 없습니다"라고 솔직히 말하세요.
3. 답변 끝에 근거로 삼은 문서를 [출처: 문서제목 › 섹션] 형식으로 표시하세요.
4. 한국어로 간결하게 답하세요.

--- HumanMessage ---
문서 발췌:
[1] (3월 러닝 기록 › 총량) 누적 거리는 96킬로미터였다.

질문: 3월에 얼마나 달렸어?


### 3-2. LCEL — 파이프로 잇기

LCEL(LangChain Expression Language)은 `|` 로 부품을 연결하는 문법입니다.

```python
chain = (
    {"context": 질문→검색→문자열, "question": RunnablePassthrough()}
    | prompt        # 변수를 채워 메시지 생성
    | llm           # 모델 호출
    | StrOutputParser()   # 응답 객체 → 문자열
)
```

각 단계가 모두 `Runnable` 이라는 같은 인터페이스를 따르기 때문에,
`invoke`(1건) · `batch`(여러 건 병렬) · `stream`(토큰 단위) 이 **공짜로** 딸려 옵니다.

In [13]:
# LCEL 체인 조립 — 검색부터 문자열 답변까지 한 줄로 흐른다
rag_chain = lc_rag.build_lcel_chain(stores[0], llm, k=3)

print("=== invoke: 한 건 ===")
print(rag_chain.invoke("3월에 총 몇 킬로미터 달렸나?"))

=== invoke: 한 건 ===


3월에 총 96킬로미터를 달렸습니다.  
[출처: 3월 러닝 기록 › 총량 / 몸 상태]


### 3-2-1. 체인 중간을 들여다보기

LCEL 은 `|` 로 이어 놓으면 편하지만, 그 대가로 **속이 보이지 않습니다.**
위 `invoke()` 가 돌려준 것은 답변 문자열 하나뿐입니다 — 검색이 무엇을 찾았는지,
프롬프트에 실제로 무엇이 들어갔는지, 시간을 어디서 썼는지는 전부 파이프 안에 묻혀 있습니다.

이게 왜 문제가 되냐면, 답이 이상할 때 **검색이 틀린 것인지 LLM 이 틀린 것인지**
구분하지 못하면 어디를 고쳐야 할지 알 수 없기 때문입니다.

| 방법 | 체인 수정 | 언제 쓰나 |
|---|---|---|
| `astream_events()` | ❌ 불필요 | 지금 돌아가는 체인을 **그대로** 관찰. 디버깅의 기본기 |
| `RunnablePassthrough.assign()` | ⭕ 필요 | 중간값이 **계속** 필요할 때(근거 표시·로그 적재·평가) |
| `ConsoleCallbackHandler` | ❌ 불필요 | 전부 덤프. 자세하지만 매우 길다 |
| LangSmith | ❌ 불필요 | 운영 추적·팀 공유 → [`M03_2_1_langsmith_hands_on.ipynb`](M03_2_1_langsmith_hands_on.ipynb) |

세 번째는 한 줄이면 됩니다(출력이 수백 줄이라 여기서는 실행하지 않습니다).

```python
from langchain_core.tracers import ConsoleCallbackHandler
rag_chain.invoke("...", config={"callbacks": [ConsoleCallbackHandler()]})
```

앞의 두 가지를 차례로 봅니다.

In [14]:
# [방법 1] 관찰 — 이미 만든 rag_chain 을 **고치지 않고** 실행 과정만 들여다본다
#
# astream_events() 는 LCEL 각 부품의 시작·끝을 이벤트로 흘려 준다.
# 비동기 함수라 await 로 부른다(동기 stream_events() 는 langchain-core 1.x 에서
# RunnableSequence 를 지원하지 않는다).
trace = await lc_rag.trace_lcel_events(rag_chain, "3월에 총 몇 킬로미터 달렸나?")

# 시간을 어디서 썼는지 — 대개 범인은 하나다
# (llm_ms 는 단계 '이름' 이 아니라 on_chat_model_end 이벤트로 잡은 값이다.
#  이름으로 찾으면 ChatPromptTemplate 이 먼저 걸리고, 모델 클래스명은 공급자마다 다르다.)
total_ms, llm_ms = trace["total_ms"], trace["llm_ms"]
print(f"\n전체 {total_ms:.0f}ms 중 LLM 구간이 {llm_ms:.0f}ms — {llm_ms / total_ms:.0%}")
print("→ 검색·포매팅을 아무리 최적화해도 체감 속도는 거의 그대로다.")
print("  체감을 바꾸려면 뒤의 stream() 셀처럼 첫 토큰을 빨리 내보내야 한다.")

# 주의: 구간을 세로로 더하면 전체 시간을 넘는다 — 단계가 줄 서서 도는 게 아니기 때문이다
overlap_sum = sum(end - begin for begin, end, _, _ in trace["steps"])
print(f"\n구간 단순합 {overlap_sum:.0f}ms > 전체 {total_ms:.0f}ms")
print("→ StrOutputParser 는 첫 토큰이 도착하자마자 열려 모델과 겹쳐 돈다.")
print("  RunnableParallel 은 retrieve·format_docs 를 감싸는 부모라 자식 구간을 포함한다.")
print("  그래서 단계별 시간은 '합계'가 아니라 **타임라인의 겹침**으로 읽어야 한다.")

[LCEL 실행 추적] 3월에 총 몇 킬로미터 달렸나?
       시작       종료  단계                       내놓은 것
------------------------------------------------------------------------------------------------
      9ms     70ms  retrieve                 3건 — 3월 러닝 기록 › 총량 / 몸 상태, 3월 러닝 기록 › 4월 목표
     69ms     72ms  format_docs              442자  '[1] (3월 러닝 기록 › 총량 / 몸 상태) 한 달 동안 열두 번 달렸고 누적 거리는 96킬로미터였다. 가장…'
      4ms     74ms  RunnableParallel<contex… {question, context}
     74ms     75ms  ChatPromptTemplate       메시지 2개 (system 185자, human 472자)


     76ms   1110ms  ChatOpenAI               토큰 40개 → 49자  '3월에 총 96킬로미터를 달렸습니다.   [출처: 3월 러닝 기록 › 총량 / 몸 상태]'
    382ms   1114ms  StrOutputParser          49자  '3월에 총 96킬로미터를 달렸습니다.   [출처: 3월 러닝 기록 › 총량 / 몸 상태]'
------------------------------------------------------------------------------------------------
  [타임라인] 총 1114ms — 막대가 겹치는 구간은 동시에 돌고 있다는 뜻이다
    retrieve               |██············································|    61ms
    format_docs            |  █···········································|     4ms
    RunnableParallel<cont… |███···········································|    70ms
    ChatPromptTemplate     |   █··········································|     1ms
    ChatOpenAI             |   ██████████████████████████████████████████·|  1034ms
    StrOutputParser        |               ███████████████████████████████|   732ms

  최종 답변: 3월에 총 96킬로미터를 달렸습니다.   [출처: 3월 러닝 기록 › 총량 / 몸 상태]

전체 1114ms 중 LLM 구간이 1034ms — 93%
→ 검색·포매팅을 아무리 최적화해도 체감 속도는 거의 그대로다.
  체감을 바꾸려

In [15]:
# [방법 2] 설계 — 중간값을 결과 dict 에 쌓아 두는 체인 (RunnablePassthrough.assign)
#
# assign() 은 "지금까지의 dict 를 그대로 흘려보내면서 키를 하나 더 붙인다"는 뜻이다.
# 그래서 마지막에 받아 보면 단계별 중간값이 전부 남아 있다 — 관찰용이 아니라
# 중간값이 계속 필요할 때(근거 표시·로그·평가) 애초에 이렇게 짓는다.
stepwise_chain = lc_rag.build_stepwise_chain(stores[0], llm, k=3)
result = stepwise_chain.invoke("무릎이 아팠을 때 어떻게 했나?")

print(f"돌려받은 키: {list(result)}\n")   # 답변만이 아니라 중간값이 전부 들어 있다
print("=" * 96)
lc_rag.print_stepwise(result)

돌려받은 키: ['question', 'hits', 'context', 'messages', 'answer']

[1] question  무릎이 아팠을 때 어떻게 했나?

[2] hits      검색 3건
      1. (0.319) 3월 러닝 기록 › 총량 / 몸 상태
      2. (0.084) 2026년 12주차 주간 업무 보고 › 이번 주 한 일 / 문제와 해결
      3. (0.081) 오사카 여행 준비 메모 › 챙길 것

[3] context   530자 — 검색 결과를 프롬프트용 문자열로 합친 것
      [1] (3월 러닝 기록 › 총량 / 몸 상태) 한 달 동안 열두 번 달렸고 누적 거리는 96킬로미터였다. 가장 길게 달린 날은 15킬로미터였고 평균 페이스는 킬로미터당 5분 40초였다.  둘째 주에 무릎이 시큰거려 사흘을 쉬었다. 신발을 바꾸고 나서는 괜찮아졌다. 아침 공복 러닝은 속이 불편해서 가벼운 식사 후에 나가는 쪽으로 바꿨다.  [2] (2026년 12주차 주간 업무 보고 › 이번 주 한 일 / 문제와 해결) 문서 수집 파이프라인의 첫 버전을 만들었다. 마크다운과 텍스트 파일을 읽어 front matter 를 분리하고 섹…

[4] messages  LLM 에 실제로 건너간 메시지 2개
      SystemMessage   185자  당신은 사용자의 개인 문서를 검색해 답하는 비서입니다.  규칙: 1. 반드시 아래 '문서 발췌'에 있는 내용만 근거로 답하세요. …
      HumanMessage    560자  문서 발췌: [1] (3월 러닝 기록 › 총량 / 몸 상태) 한 달 동안 열두 번 달렸고 누적 거리는 96킬로미터였다. 가장 길게…

[5] answer    무릎이 시큰거렸을 때는 사흘간 쉬고, 그 후에 신발을 바꾸어 통증을 완화시켰습니다.  
[출처: 3월 러닝 기록 › 총량 / 몸 상태]


**여기서 확인할 것**

- **`[2] hits` 의 점수 분포** — 1위만 높고 2·3위가 뚝 떨어졌다면 쓸 만한 근거는 사실상 1건이고,
  나머지는 프롬프트 길이만 늘린 셈입니다. `k` 를 줄일 신호입니다.
- **`[4] messages` 가 LLM 이 본 전부입니다.** 답변이 이상하면 여기부터 보세요.
  여기에 근거가 없는데 그럴듯한 답이 나왔다면 그게 바로 환각입니다.
- **타임라인의 겹침** — LLM 구간이 전체의 대부분을 먹습니다. 검색을 최적화해도 체감은 안 바뀝니다.
  그리고 단계별 시간을 **더하지 마세요**. `StrOutputParser` 는 모델이 아직 토큰을 뱉는 중에
  이미 열려 있고, `RunnableParallel` 은 자식을 감싸는 부모라 구간이 중복됩니다 —
  LCEL 은 순차 실행이 아니라 **겹쳐서 흐르는** 파이프입니다.

> **디버깅 순서**: 답이 틀렸을 때 `[2] hits` → `[4] messages` → 답변 순으로 봅니다.
> 근거가 안 잡혔으면 **검색 문제**(청킹·`k`·임베딩), 근거는 맞는데 답이 틀렸으면
> **LLM·프롬프트 문제** 입니다. 이 구분을 먼저 하지 않으면 엉뚱한 곳을 고치게 됩니다.

In [16]:
# batch: 여러 질문을 한 번에 (내부적으로 병렬 처리된다)
import time

questions = ["오사카에서 미리 예약할 것은?",
             "대외비 문서를 외부로 보내면 어떻게 되나?",
             "RunnableLambda 는 언제 쓰나?"]

started = time.perf_counter()
answers = rag_chain.batch(questions)
print(f"batch {len(questions)}건 — {time.perf_counter() - started:.1f}s\n")
for question, answer in zip(questions, answers):
    print(f"Q. {question}")
    print(f"A. {answer.strip()[:180]}\n")

batch 3건 — 3.0s

Q. 오사카에서 미리 예약할 것은?
A. 오사카에서는 간사이 공항에서 시내로 가는 특급 열차표를 미리 끊어 두고, 인기 있는 식당도 사전 예약하는 것이 좋습니다.  
[출처: 오사카 여행 준비 메모 › 일정 / 예약할 것]

Q. 대외비 문서를 외부로 보내면 어떻게 되나?
A. 대외비 문서를 외부로 전송하면 즉시 보고 대상이며 감사 기록이 남는다.  
[출처: 사내 문서 보안 지침 › 위반 시]

Q. RunnableLambda 는 언제 쓰나?
A. RunnableLambda는 평범한 파이썬 함수를 LangChain 체인 안으로 끌어들일 때 사용합니다.  
[출처: LangChain LCEL 학습 노트 › 자주 쓰는 조각 / 실수했던 것]



In [31]:
# stream: 토큰이 생성되는 대로 흘려보낸다 (체감 응답속도가 크게 좋아진다)
print("stream 출력:\n")
for token in rag_chain.stream("무릎이 아팠을 때 어떻게 했나?"):
    print(token, end="", flush=True)
print()

stream 출력:

무릎이 시큰거려 사흘을 쉬었고, 신발을 바꾼 뒤에는 괜찮아졌습니다.  
[출처: 3월 러닝 기록 › 총량 / 몸 상태]


In [18]:
# 근거까지 함께 보는 체인 — RAG 는 '무엇을 보고 답했는가'가 답변만큼 중요하다
traced_chain = lc_rag.build_traced_chain(stores[0], llm, k=3)

for question in ["킥오프에서 권한에 대해 무엇을 정했나?",
                 "청크 크기 문제를 어떻게 해결했나?"]:
    lc_rag.print_chain_result(traced_chain.invoke(question))

질문: 킥오프에서 권한에 대해 무엇을 정했나?

검색된 근거 3건:
  [1] (0.307) 사내 문서 검색 시스템 킥오프 회의록 › 결정 사항
  [2] (0.241) 오사카 여행 준비 메모 › 일정 / 예약할 것
  [3] (0.230) 오사카 여행 준비 메모 › 챙길 것

답변:
문서 권한이 부서별로 다르므로 메타데이터에 부서와 공개 범위를 반드시 넣기로 결정했습니다.  
[출처: 사내 문서 검색 시스템 킥오프 회의록 › 결정 사항]
------------------------------------------------------------------------------------


질문: 청크 크기 문제를 어떻게 해결했나?

검색된 근거 3건:
  [1] (0.313) 2026년 12주차 주간 업무 보고 › 이번 주 한 일 / 문제와 해결
  [2] (0.286) 3월 러닝 기록 › 총량 / 몸 상태
  [3] (0.238) 3월 러닝 기록 › 4월 목표

답변:
청크 크기를 키우고, 청크 사이에 겹침을 두는 방식으로 바꾸어 맥락을 유지하도록 했습니다.  
[출처: 2026년 12주차 주간 업무 보고 › 이번 주 한 일 / 문제와 해결]
------------------------------------------------------------------------------------


---
## 4. RAG Agent — 검색을 '도구'로 넘기기

LCEL 체인은 **무조건 1회 검색 후 답변** 합니다. 빠르고 예측 가능하지만 한계가 있습니다.

| 질문 유형 | LCEL 체인 | RAG Agent |
|---|---|---|
| "3월에 몇 km 달렸어?" | ✅ 1회 검색으로 충분 | ✅ 되지만 느림 |
| "킥오프에서 정한 기간과, 주간보고의 청크 문제는?" | ❌ 한 번의 검색으로 두 문서를 못 잡음 | ✅ 검색어를 바꿔 2회 검색 |
| "이 문서에 없는 내용" | ❌ 억지로 답할 위험 | ✅ 다시 찾아보고 없다고 판단 |

Agent 는 검색을 **도구(tool)** 로 받아, **언제 · 몇 번 · 어떤 검색어로** 찾을지 스스로 정합니다.
LangChain 1.x 의 `create_agent()` 를 씁니다(내부적으로 LangGraph 로 동작).

> ⚠️ 도구 호출을 지원하는 모델이 필요합니다. 지원하지 않는 공급자면 아래 셀이 안내 후 건너뜁니다.

In [32]:
# 검색 도구의 정체 — 도구 설명(description)이 곧 LLM 이 읽는 사용 설명서다
search_tool = lc_rag.make_search_tool(stores[0], k=3)

print(f"도구 이름 : {search_tool.name}")
print(f"입력 스키마: {search_tool.args}")
print(f"\n설명(LLM 이 읽는 부분):\n{search_tool.description}")

print("\n=== 도구를 직접 호출해 보면 ===")
print(search_tool.invoke({"query": "러닝 누적 거리"})[:300])

도구 이름 : search_personal_docs
입력 스키마: {'query': {'title': 'Query', 'type': 'string'}}

설명(LLM 이 읽는 부분):
개인 문서(회의록·업무보고·학습노트·사내규정·개인메모)를 시맨틱 검색한다. 사용자의 기록·일정·결정사항·수치를 묻는 질문이면 반드시 먼저 호출할 것. 결과가 부족하면 검색어를 바꿔 여러 번 호출해도 된다.

=== 도구를 직접 호출해 보면 ===
[1] (3월 러닝 기록 › 총량 / 몸 상태) 한 달 동안 열두 번 달렸고 누적 거리는 96킬로미터였다. 가장 길게 달린 날은 15킬로미터였고 평균 페이스는 킬로미터당 5분 40초였다.  둘째 주에 무릎이 시큰거려 사흘을 쉬었다. 신발을 바꾸고 나서는 괜찮아졌다. 아침 공복 러닝은 속이 불편해서 가벼운 식사 후에 나가는 쪽으로 바꿨다.

[2] (사내 문서 검색 시스템 킥오프 회의록 › 결정 사항) 1. 1차 목표는 사내 규정 문서와 회의록을 대상으로 하는 시맨틱 검색이다. 2. 임베딩은 사내망에서 동작해야 하므로 오프라인 모델을


In [33]:
# RAG Agent 실행 — 검색 횟수와 검색어를 모델이 스스로 정한다
try:
    agent = lc_rag.build_rag_agent(llm, stores[0], k=3)

    # 단일 문서로 답할 수 있는 질문
    lc_rag.run_agent(agent, "3월 러닝 기록에서 가장 길게 달린 거리는?")

    # 두 문서를 엮어야 하는 질문 — 여기서 Agent 의 반복 검색이 값을 한다
    lc_rag.run_agent(agent, "킥오프에서 정한 프로토타입 기간과, "
                            "주간보고에 적힌 청크 문제는 각각 무엇이었나?")
except Exception as e:
    print(f"에이전트 실행 실패: {type(e).__name__}: {str(e)[:200]}")
    print("→ 도구 호출을 지원하는 공급자로 .env 의 LLM_PROVIDER 를 바꿔 보세요")

질문: 3월 러닝 기록에서 가장 길게 달린 거리는?
  🔍 검색 1회차: "3월 러닝 기록 거리"

답변:
3월 러닝 기록에서 가장 길게 달린 거리는 **15킬로미터**입니다.
------------------------------------------------------------------------------------
질문: 킥오프에서 정한 프로토타입 기간과, 주간보고에 적힌 청크 문제는 각각 무엇이었나?
  🔍 검색 1회차: "킥오프 프로토타입 기간"
  🔍 검색 2회차: "주간보고 청크 문제"

답변:
- **킥오프에서 정한 프로토타입 기간**: 6주 안에 프로토타입을 만들어 사내 데모를 진행한다.  
- **주간보고에 적힌 청크 문제**: 청크를 너무 잘게 쪼개서 검색 결과가 문장 조각처럼 나와 답변 품질이 떨어졌다. (해결 방안: 청크 크기를 키우고 겹침을 두어 맥락을 유지함)
------------------------------------------------------------------------------------


In [35]:
# 같은 질문을 체인과 에이전트에 각각 물어 속도·품질을 비교한다
lc_rag.compare_chain_and_agent(stores[0], llm, [
    "오사카 여행에서 챙길 물건은?",                        # 단순 — 체인으로 충분
    "회의록의 결정 사항 중 보안 지침과 관련된 것은?",        # 복합 — 두 문서를 엮어야 함
], k=3)


질문: 오사카 여행에서 챙길 물건은?

[LCEL 체인] 2.0s (검색 1회 고정)
  오사카 여행 시 챙길 물건은 110볼트 변환 어댑터, 우산, 교통카드, 그리고 현금 사용이 많은 가게를 대비해 엔화를 조금 준비하는 것이 좋습니다.  
[출처: 오사카 여행 준비 메모 › 챙길 것]

[RAG Agent] 2.8s / 검색 1회 ('오사카 여행 챙길 물건')
  오사카 여행 시 챙겨야 할 물건은 다음과 같습니다.

- 110볼트 변환 어댑터  
- 우산  
- 교통카드 (ICOCA 등)  
- 엔화 (현금이 필요한 가게가 많아 소량 준비)  

이 내용은 사용자의 오사카 여행 준비 메모에 명시된 “챙길 것” 항목을 바탕으로 답변했습니다.

질문: 회의록의 결정 사항 중 보안 지침과 관련된 것은?

[LCEL 체인] 1.9s (검색 1회 고정)
  회의록에서 보안 지침과 관련된 결정 사항은 **기획팀이 부서별 문서 접근 정책을 정리해 공유하는 것**입니다.  
[출처: 사내 문서 검색 시스템 킥오프 회의록 › 다음 액션]

[RAG Agent] 8.6s / 검색 2회 ('회의록 결정 사항 보안 지침', '회의록 결정 사항 보안 지침')
  회의록에서 보안 지침과 관련된 결정 사항은 다음과 같습니다.

1. **대외비 문서 유출 시 조치**  
   - 대외비 문서를 외부로 전송하면 즉시 보고 대상이며, 감사 기록이 남는다.

2. **문서 등급 구분 및 검색 시스템 적용 원칙**  
   - 문서는 **공개, 사내한정, 대외비** 세 등급으로 나눈다.  
   - 등급은 문서를 만든 사람이 지정하며, 지정하지 않으면 **사내한정**으로 간주한다.  
   - 검색 색인에는 등급 정보를 반드시 함께


### 4-2. 에이전트의 함정 — 검색을 건너뛰면 그냥 LLM 이다

위 비교에서 **검색 횟수** 를 반드시 확인하세요. 실행 결과에 따라 이런 일이 일어납니다.

- 첫 질문("챙길 물건")에서 에이전트가 **검색 0회** 로 답하는 경우가 있습니다.
  모델이 "여행 준비물쯤은 내가 안다"고 판단한 것입니다. 그러면 답변은 그럴듯하지만
  **내 문서에 없는 일반론**(여권·보조배터리 …)이 됩니다. 반면 LCEL 체인은 항상 검색하므로
  문서에 적힌 그대로(110볼트 어댑터·우산·교통카드) 답합니다.
- 이것이 **에이전트에게 자유를 준 대가** 입니다. 검색 여부를 모델이 정한다는 것은
  검색하지 않을 자유도 준다는 뜻입니다.

**대응 방법**

| 방법 | 설명 |
|---|---|
| 시스템 프롬프트 강화 | "반드시 도구를 먼저 호출하라"를 명시(이미 적용했지만 작은 모델은 무시하기도 함) |
| 도구 강제 | `tool_choice` 로 첫 턴에 도구 호출을 강제 |
| 하이브리드 | 1차는 LCEL 체인, "찾을 수 없음"일 때만 에이전트로 승격 |
| 검증 | `invoke_agent()` 가 돌려주는 검색어 목록이 비었으면 답변을 신뢰하지 않음 |

**결론**: 사내 문서 Q&A 처럼 **근거가 반드시 문서여야 하는** 용도라면 기본은 LCEL 체인입니다.
에이전트는 여러 문서를 엮어야 하는 복합 질문에 한정해 쓰고, 검색 횟수를 로그로 남기세요.

---
## 5. 정리

### 배운 것

| 절 | 내용 | 핵심 API |
|---|---|---|
| 1 | 문서 구조화 · 청킹 4전략 · 메타데이터 설계 | `doc_prep.load_markdown_docs()`, `compare_chunking()`, `build_chunk_metadata()` |
| 1-2 | 청킹 전략을 **검색 정확도로** 선택 | `vs.evaluate_chunking()` |
| 2 | ChromaDB · FAISS · Qdrant 비교 + 메타데이터 필터 | `vs.get_vector_store()`, `compare_search()` |
| 2-2-1 | FAISS 후처리 필터가 **과다인출에 의존** 함을 측정 | `vs.faiss_overfetch_demo()`, `search(..., overfetch=n)` |
| 3 | PromptTemplate → LCEL 체인(invoke/batch/stream) | `lc_rag.build_rag_prompt()`, `build_lcel_chain()` |
| 3-2-1 | 체인 **중간 실행 결과** 들여다보기 | `lc_rag.trace_lcel_events()`, `build_stepwise_chain()` |
| 4 | 검색을 도구로 준 RAG Agent | `lc_rag.make_search_tool()`, `build_rag_agent()` |

### 기억할 원칙

1. **RAG 품질은 전처리에서 갈린다** — 임베딩 모델을 바꾸기 전에 청킹부터 점검한다.
2. **청킹 전략은 측정해서 고른다** — 청크 수·평균 길이가 아니라 **검색 정확도** 가 기준이다.
3. **메타데이터 없는 청크는 쓸모가 절반** — 출처를 못 대면 답변을 검증할 수 없다.
   같은 이유로 **체인 중간을 볼 수 없으면 고칠 수도 없다** — 검색이 틀렸는지 LLM 이 틀렸는지
   구분하는 것이 RAG 디버깅의 출발점이다(`trace_lcel_events()`).
4. **벡터 DB 는 같은 벡터면 같은 결과** — 차이는 필터링·운영 기능·규모에서 난다.
   특히 **후처리 필터(FAISS)는 조용히 결과를 누락** 시킨다. "결과 없음"이 정말 없는 것인지
   덜 뽑아서 못 본 것인지 구분되지 않으므로, 필터가 중요하면 인덱스 통합형(Qdrant)을 쓴다.
5. **체인 먼저, 에이전트는 필요할 때** — 단순 질문에 에이전트를 쓰면 느리기만 하고,
   최악의 경우 **검색을 건너뛰고 환각** 한다. 검색 횟수를 반드시 확인한다.

### 다음 편

- [`M04_3_graph_rag.ipynb`](M04_3_graph_rag.ipynb) — 지식 그래프 · Neo4j 벡터 인덱스 · LlamaIndex GraphRAG
- [`M04_4_agent_memory.ipynb`](M04_4_agent_memory.ipynb) — 에이전트 메모리를 직접 구현해 RAG 와 결합
- [`M04_5_memory.ipynb`](M04_5_memory.ipynb) — 같은 메모리를 LangGraph 표준 부품으로(체크포인터 · Store · 영속화)